In [ ]:
import torch
import evaluate
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import TrainingArguments, Trainer, XLMRobertaModel, XLMRobertaTokenizerFast, XLMRobertaForTokenClassification
from transformers import get_linear_schedule_with_warmup, EarlyStoppingCallback, AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
import torch.nn as nn
import pandas as pd
import numpy as np
#from sklearn.metrics import classification_report
from seqeval.metrics import f1_score, classification_report, accuracy_score, precision_score, recall_score
import random
from tqdm import tqdm
import ast
import os

In [ ]:
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

In [ ]:
!pip install torch
!pip install transformers
!pip install seqeval
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 104.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [ ]:
def set_random_seed(seed):
    torch.backends.cudnn.deterministic = True
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

In [ ]:
SEED = 42
set_random_seed(SEED) # seed LOCKED !!!!!!!!!!!!

PLO sorted data

In [ ]:
df = pd.read_csv('plomatch_iob_withempty.csv')
df['translated_text'] = df['translated_text'].apply(ast.literal_eval)
df['iob'] = df['iob'].apply(ast.literal_eval)

In [ ]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
mismatch = train_df[train_df['translated_text'].str.len() != train_df['iob'].str.len()]
print(f'Mismatches found: {len(mismatch)}')

Mismatches found: 0


In [ ]:
train_df = train_df[train_df['translated_text'].str.len() == train_df['iob'].str.len()]
val_df = val_df[val_df['translated_text'].str.len() == val_df['iob'].str.len()]

In [ ]:
train_dataset = Dataset.from_pandas(train_df[['translated_text', 'iob']])
val_dataset = Dataset.from_pandas(val_df[['translated_text', 'iob']])

In [ ]:
tokenizer = XLMRobertaTokenizerFast.from_pretrained('xlm-roberta-base')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

In [ ]:
label_list = ['O', 'B-LOC', 'I-LOC', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG']
label_to_id = {l: i for i, l in enumerate(label_list)}
id_to_label = {i: l for l, i in label_to_id.items()}

In [ ]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example['translated_text'],
        is_split_into_words=True,
        truncation=True,
        padding='max_length',
        max_length=128
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            labels.append(label_to_id[example['iob'][word_idx]])
        else:
            labels.append(-100)
        previous_word_idx = word_idx

    tokenized_inputs['labels'] = labels
    return tokenized_inputs

In [ ]:
train_dataset = train_dataset.map(tokenize_and_align_labels)
val_dataset = val_dataset.map(tokenize_and_align_labels)

Map:   0%|          | 0/20519 [00:00<?, ? examples/s]

Map:   0%|          | 0/5134 [00:00<?, ? examples/s]

In [ ]:
def compute_metrics(p):
    predictions, labels = p
    preds = np.argmax(predictions, axis=2)

    true_labels = []
    true_preds = []

    for pred, label in zip(preds, labels):
        temp_pred = []
        temp_label = []
        for p, l in zip(pred, label):
            if l != -100:
                temp_label.append(id_to_label[l])
                temp_pred.append(id_to_label[p])
        true_labels.append(temp_label)
        true_preds.append(temp_pred)

    report = classification_report(true_labels, true_preds, output_dict=True)

    return {
        'accuracy': accuracy_score(true_labels, true_preds),  # entity-level accuracy
        'f1': f1_score(true_labels, true_preds),
        'precision': report['macro avg']['precision'],
        'recall': report['macro avg']['recall']
    }

In [ ]:
model = XLMRobertaForTokenClassification.from_pretrained(
    'xlm-roberta-base',
    num_labels=len(label_list),
    id2label=id_to_label,
    label2id=label_to_id
)

training_args = TrainingArguments(
    output_dir='./ner-xlm-roberta',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-19-a3e8f24f4de3>:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sol-arina (sol-arina-russian-state-university-for-the-humanities) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.029500,0.017571,0.995156,0.919092,0.899472,0.909000
2,0.010000,0.010883,0.997180,0.950710,0.938221,0.950779
3,0.005800,0.011279,0.997525,0.959834,0.948949,0.962993


TrainOutput(global_step=3849, training_loss=0.024505978839742454, metrics={'train_runtime': 2089.8587, 'train_samples_per_second': 29.455, 'train_steps_per_second': 1.842, 'total_flos': 4021342888958208.0, 'train_loss': 0.024505978839742454, 'epoch': 3.0})

In [ ]:
trainer.compute_metrics = compute_metrics
metrics = trainer.evaluate()
print(metrics)

{'eval_loss': 0.011279039084911346, 'eval_accuracy': 0.997525440960666, 'eval_f1': 0.9598342365317182, 'eval_precision': 0.9489489285819377, 'eval_recall': 0.9629928860585321, 'eval_runtime': 35.857, 'eval_samples_per_second': 143.18, 'eval_steps_per_second': 8.952, 'epoch': 3.0}


In [ ]:
def compute_metrics_noO(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred, label in zip(predictions, labels):
        sentence_preds = []
        sentence_labels = []

        for p_id, l_id in zip(pred, label):
            if l_id == -100:
                continue
            sentence_preds.append(id_to_label[p_id])
            sentence_labels.append(id_to_label[l_id])

        true_predictions.append(sentence_preds)
        true_labels.append(sentence_labels)

    report = classification_report(true_labels, true_predictions, output_dict=True, zero_division=0)

    results = {}
    for label, scores in report.items():
        if label == 'O' or label in ['accuracy', 'macro avg', 'weighted avg']:
            continue
        results[f'{label}_f1'] = scores['f1-score']
        results[f'{label}_precision'] = scores['precision']
        results[f'{label}_recall'] = scores['recall']

    # без 'O'
    f1_scores = [v for k, v in results.items() if k.endswith('_f1')]
    results['macro_f1_excl_O'] = np.mean(f1_scores)

    return results

In [ ]:
trainer.compute_metrics = compute_metrics_noO
metrics = trainer.evaluate()
print(metrics)

{'eval_loss': 0.011279039084911346, 'eval_LOC_f1': 0.945617402431222, 'eval_LOC_precision': 0.928391959798995, 'eval_LOC_recall': 0.9634941329856584, 'eval_ORG_f1': 0.9549738219895288, 'eval_ORG_precision': 0.9519832985386222, 'eval_ORG_recall': 0.957983193277311, 'eval_PER_f1': 0.9669861554845581, 'eval_PER_precision': 0.9664715274081959, 'eval_PER_recall': 0.9675013319126265, 'eval_micro avg_f1': 0.9598342365317182, 'eval_micro avg_precision': 0.9546607482561826, 'eval_micro avg_recall': 0.9650641025641026, 'eval_macro_f1_excl_O': 0.9568529041092567, 'eval_runtime': 37.1926, 'eval_samples_per_second': 138.038, 'eval_steps_per_second': 8.631, 'epoch': 3.0}


all predictions

In [ ]:
predictions_output = trainer.predict(val_dataset)
raw_preds = predictions_output.predictions
true_labels = predictions_output.label_ids

predicted_ids = np.argmax(raw_preds, axis=2)

In [ ]:
predictions_text = []
labels_text = []

for preds, labels in zip(predicted_ids, true_labels):
    pred_seq = []
    label_seq = []
    for p, l in zip(preds, labels):
        if l != -100:
            pred_seq.append(id_to_label[p])
            label_seq.append(id_to_label[l])
    predictions_text.append(pred_seq)
    labels_text.append(label_seq)

In [ ]:
results_df = pd.DataFrame({
    'true_labels': labels_text,
    'predicted_labels': predictions_text
})

results_df.to_csv('ner_predictionsPLOfiltered.csv', index=False) # можно на них посмотреть

In [ ]:
model.save_pretrained('ner-xlm-besksl-PLOfiltered')
tokenizer.save_pretrained('ner-xlm-besksl-PLOfiltered')

('ner-xlm-besksl-PLOfiltered/tokenizer_config.json',
 'ner-xlm-besksl-PLOfiltered/special_tokens_map.json',
 'ner-xlm-besksl-PLOfiltered/sentencepiece.bpe.model',
 'ner-xlm-besksl-PLOfiltered/added_tokens.json',
 'ner-xlm-besksl-PLOfiltered/tokenizer.json')

kmeans augmented

In [ ]:
df_k = pd.read_csv('kmeans_iob_withempty.csv')
df_k['translated_text'] = df_k['translated_text'].apply(ast.literal_eval)
df_k['iob'] = df_k['iob'].apply(ast.literal_eval)

In [ ]:
train_df, val_df = train_test_split(df_k, test_size=0.2, random_state=42)

In [ ]:
mismatch = train_df[train_df['translated_text'].str.len() != train_df['iob'].str.len()]
print(f'Mismatches found: {len(mismatch)}')

Mismatches found: 0


In [ ]:
train_df = train_df[train_df['translated_text'].str.len() == train_df['iob'].str.len()]
val_df = val_df[val_df['translated_text'].str.len() == val_df['iob'].str.len()]

In [ ]:
train_dataset = Dataset.from_pandas(train_df[['translated_text', 'iob']])
val_dataset = Dataset.from_pandas(val_df[['translated_text', 'iob']])

In [ ]:
train_dataset = train_dataset.map(tokenize_and_align_labels)
val_dataset = val_dataset.map(tokenize_and_align_labels)

Map:   0%|          | 0/9373 [00:00<?, ? examples/s]

Map:   0%|          | 0/2339 [00:00<?, ? examples/s]

In [ ]:
model = XLMRobertaForTokenClassification.from_pretrained(
    'xlm-roberta-base',
    num_labels=len(label_list),
    id2label=id_to_label,
    label2id=label_to_id
)

training_args = TrainingArguments(
    output_dir='./ner-xlm-roberta',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-35-a3e8f24f4de3>:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.068000,0.013342,0.996290,0.725166,0.823550,0.620879
2,0.010700,0.011499,0.997040,0.847594,0.781135,0.949765
3,0.004500,0.006204,0.998499,0.926619,0.896585,0.970163


TrainOutput(global_step=1758, training_loss=0.024105435040216804, metrics={'train_runtime': 1209.3896, 'train_samples_per_second': 23.251, 'train_steps_per_second': 1.454, 'total_flos': 1836933909947136.0, 'train_loss': 0.024105435040216804, 'epoch': 3.0})

In [ ]:
trainer.compute_metrics = compute_metrics
metrics = trainer.evaluate()
print(metrics)

{'eval_loss': 0.006204061210155487, 'eval_accuracy': 0.9984987561122073, 'eval_f1': 0.9266187050359712, 'eval_precision': 0.8965852130325814, 'eval_recall': 0.9701630085996663, 'eval_runtime': 16.2, 'eval_samples_per_second': 144.383, 'eval_steps_per_second': 9.074, 'epoch': 3.0}


In [ ]:
model.save_pretrained('ner-xlm-besksl-KMEANSfiltered')
tokenizer.save_pretrained('ner-xlm-besksl-KMEANSfiltered')

('ner-xlm-besksl-KMEANSfiltered/tokenizer_config.json',
 'ner-xlm-besksl-KMEANSfiltered/special_tokens_map.json',
 'ner-xlm-besksl-KMEANSfiltered/sentencepiece.bpe.model',
 'ner-xlm-besksl-KMEANSfiltered/added_tokens.json',
 'ner-xlm-besksl-KMEANSfiltered/tokenizer.json')

In [ ]:
trainer.compute_metrics = compute_metrics_noO
metrics = trainer.evaluate()
print(metrics)

{'eval_loss': 0.006204061210155487, 'eval_LOC_f1': 0.8857142857142859, 'eval_LOC_precision': 0.8303571428571429, 'eval_LOC_recall': 0.9489795918367347, 'eval_ORG_f1': 0.9801324503311258, 'eval_ORG_precision': 0.9736842105263158, 'eval_ORG_recall': 0.9866666666666667, 'eval_PER_f1': 0.9281437125748503, 'eval_PER_precision': 0.8857142857142857, 'eval_PER_recall': 0.9748427672955975, 'eval_micro avg_f1': 0.9266187050359712, 'eval_micro avg_precision': 0.8870523415977961, 'eval_micro avg_recall': 0.9698795180722891, 'eval_macro_f1_excl_O': 0.9301522884140583, 'eval_runtime': 16.7523, 'eval_samples_per_second': 139.622, 'eval_steps_per_second': 8.775, 'epoch': 3.0}


предикты

In [ ]:
predictions_output = trainer.predict(val_dataset)
raw_preds = predictions_output.predictions
true_labels = predictions_output.label_ids

predicted_ids = np.argmax(raw_preds, axis=2)

In [ ]:
predictions_text = []
labels_text = []

for preds, labels in zip(predicted_ids, true_labels):
    pred_seq = []
    label_seq = []
    for p, l in zip(preds, labels):
        if l != -100:
            pred_seq.append(id_to_label[p])
            label_seq.append(id_to_label[l])
    predictions_text.append(pred_seq)
    labels_text.append(label_seq)

In [ ]:
results_df = pd.DataFrame({
    'true_labels': labels_text,
    'predicted_labels': predictions_text
})

results_df.to_csv('ner_predictionsKMEANSfiltered.csv', index=False) # можно на них посмотреть

DBSCAN augmented

In [ ]:
df_d = pd.read_csv('dbscan_iob_withempty.csv')
df_d['translated_text'] = df_d['translated_text'].apply(ast.literal_eval)
df_d['iob'] = df_d['iob'].apply(ast.literal_eval)

In [ ]:
train_df, val_df = train_test_split(df_d, test_size=0.2, random_state=42)

In [ ]:
mismatch = train_df[train_df['translated_text'].str.len() != train_df['iob'].str.len()]
print(f'Mismatches found: {len(mismatch)}')

Mismatches found: 0


In [ ]:
train_df = train_df[train_df['translated_text'].str.len() == train_df['iob'].str.len()]
val_df = val_df[val_df['translated_text'].str.len() == val_df['iob'].str.len()]

In [ ]:
train_dataset = Dataset.from_pandas(train_df[['translated_text', 'iob']])
val_dataset = Dataset.from_pandas(val_df[['translated_text', 'iob']])

In [ ]:
train_dataset = train_dataset.map(tokenize_and_align_labels)
val_dataset = val_dataset.map(tokenize_and_align_labels)

Map:   0%|          | 0/13185 [00:00<?, ? examples/s]

Map:   0%|          | 0/3284 [00:00<?, ? examples/s]

In [ ]:
model = XLMRobertaForTokenClassification.from_pretrained(
    'xlm-roberta-base',
    num_labels=len(label_list),
    id2label=id_to_label,
    label2id=label_to_id
)

training_args = TrainingArguments(
    output_dir='./ner-xlm-roberta',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-56-a3e8f24f4de3>:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.104800,0.021808,0.994102,0.854911,0.747689,0.895234
2,0.010800,0.010659,0.997341,0.943323,0.897911,0.973119
3,0.005800,0.007723,0.998090,0.956009,0.931294,0.972032


TrainOutput(global_step=2475, training_loss=0.030116858337864733, metrics={'train_runtime': 1313.2943, 'train_samples_per_second': 30.119, 'train_steps_per_second': 1.885, 'total_flos': 2584015107505920.0, 'train_loss': 0.030116858337864733, 'epoch': 3.0})

In [ ]:
trainer.compute_metrics = compute_metrics
metrics = trainer.evaluate()
print(metrics)

{'eval_loss': 0.007722815033048391, 'eval_accuracy': 0.9980901159681584, 'eval_f1': 0.9560094265514533, 'eval_precision': 0.9312944425809211, 'eval_recall': 0.9720317938981031, 'eval_runtime': 22.7776, 'eval_samples_per_second': 144.177, 'eval_steps_per_second': 9.044, 'epoch': 3.0}


In [ ]:
model.save_pretrained('ner-xlm-besksl-DBSCANfiltered')
tokenizer.save_pretrained('ner-xlm-besksl-DBSCANfiltered')

('ner-xlm-besksl-DBSCANfiltered/tokenizer_config.json',
 'ner-xlm-besksl-DBSCANfiltered/special_tokens_map.json',
 'ner-xlm-besksl-DBSCANfiltered/sentencepiece.bpe.model',
 'ner-xlm-besksl-DBSCANfiltered/added_tokens.json',
 'ner-xlm-besksl-DBSCANfiltered/tokenizer.json')

In [ ]:
trainer.compute_metrics = compute_metrics_noO
metrics = trainer.evaluate()
print(metrics)

{'eval_loss': 0.007722815033048391, 'eval_LOC_f1': 0.923076923076923, 'eval_LOC_precision': 0.8888888888888888, 'eval_LOC_recall': 0.96, 'eval_ORG_f1': 0.9662447257383966, 'eval_ORG_precision': 0.9581589958158996, 'eval_ORG_recall': 0.9744680851063829, 'eval_PER_f1': 0.963917525773196, 'eval_PER_precision': 0.9468354430379747, 'eval_PER_recall': 0.9816272965879265, 'eval_micro avg_f1': 0.9560094265514533, 'eval_micro avg_precision': 0.9368745188606621, 'eval_micro avg_recall': 0.9759422614274258, 'eval_macro_f1_excl_O': 0.9523121502849922, 'eval_runtime': 23.064, 'eval_samples_per_second': 142.386, 'eval_steps_per_second': 8.932, 'epoch': 3.0}


predict

In [ ]:
predictions_output = trainer.predict(val_dataset)
raw_preds = predictions_output.predictions
true_labels = predictions_output.label_ids

predicted_ids = np.argmax(raw_preds, axis=2)

In [ ]:
predictions_text = []
labels_text = []

for preds, labels in zip(predicted_ids, true_labels):
    pred_seq = []
    label_seq = []
    for p, l in zip(preds, labels):
        if l != -100:
            pred_seq.append(id_to_label[p])
            label_seq.append(id_to_label[l])
    predictions_text.append(pred_seq)
    labels_text.append(label_seq)

In [ ]:
results_df = pd.DataFrame({
    'true_labels': labels_text,
    'predicted_labels': predictions_text
})

results_df.to_csv('ner_predictionsDBSCANfiltered.csv', index=False) # можно на них посмотреть